In [22]:
import random
import pickle
from collections import defaultdict
from typing import List, Tuple, Optional

# ---------------- TicTacToe Environment ----------------
class TicTacToe:
    WIN_LINES = [
        (0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)
    ]
    def __init__(self):
        self.reset()
    def reset(self):
        self.board = [0]*9
        self.current_player = 1
        self.last_move = None
        return self.get_state()
    def get_state(self) -> Tuple[int,...]:
        return tuple(self.board)
    def available_actions(self) -> List[int]:
        return [i for i,v in enumerate(self.board) if v==0]
    def step(self, action:int, player:int) -> Tuple[Tuple[int,...],int,bool,Optional[int]]:
        if self.board[action]!=0:
            raise ValueError(f"Invalid action {action}")
        self.board[action]=player
        self.last_move=(action,player)
        winner=self.check_winner()
        if winner is not None:
            done=True
            reward=1 if winner==player else -1 if winner!=0 else 0
            return self.get_state(),reward,done,winner
        if self.is_draw():
            return self.get_state(),0,True,0
        return self.get_state(),0,False,None
    def check_winner(self) -> Optional[int]:
        for a,b,c in self.WIN_LINES:
            s=self.board[a]+self.board[b]+self.board[c]
            if s==3: return 1
            if s==-3: return -1
        return None
    def is_draw(self) -> bool:
        return all(v!=0 for v in self.board) and self.check_winner() is None
    def render(self):
        symbols={1:'X',-1:'O',0:' '}
        rows=[self.board[i:i+3] for i in range(0,9,3)]
        for r in rows:
            print(' | '.join(symbols[c] for c in r))
        print()

# ---------------- Q-Learning Agent ----------------
class QLearningAgent:
    def __init__(self,name='QAgent',alpha=0.3,gamma=0.9,epsilon=0.1):
        self.name=name
        self.alpha=alpha
        self.gamma=gamma
        self.epsilon=epsilon
        self.Q=defaultdict(lambda:[0.0]*9)
    def get_qs(self,state:Tuple[int,...])->List[float]:
        return self.Q[state]
    def choose_action(self,state:Tuple[int,...],available_actions:List[int],explore:bool=True)->int:
        if explore and random.random()<self.epsilon:
            return random.choice(available_actions)
        qs=self.get_qs(state)
        best_value=max(qs[a] for a in available_actions)
        best_actions=[a for a in available_actions if qs[a]==best_value]
        return random.choice(best_actions)
    def update(self,state:Tuple[int,...],action:int,reward:float,next_state:Tuple[int,...],
               next_available:List[int],done:bool):
        q_values=self.get_qs(state)
        q_predict=q_values[action]
        if done:
            q_target=reward
        else:
            next_qs=self.get_qs(next_state)
            q_target=reward+self.gamma*max(next_qs[a] for a in next_available) if next_available else reward
        q_values[action]+=self.alpha*(q_target-q_predict)
    def save(self,filename:str):
        with open(filename,'wb') as f:
            pickle.dump(dict(self.Q),f)
    def load(self,filename:str):
        with open(filename,'rb') as f:
            qdict=pickle.load(f)
            self.Q=defaultdict(lambda:[0.0]*9,qdict)

# ---------------- Training Function ----------------
def train(agent:QLearningAgent,episodes:int=20000,opponent:str='random'):
    env=TicTacToe()
    stats={'wins':0,'losses':0,'draws':0}
    for ep in range(1,episodes+1):
        state=env.reset()
        done=False
        player=1
        history=[]
        while not done:
            if player==1:
                avail=env.available_actions()
                action=agent.choose_action(state,avail,explore=True)
                next_state,reward,done,winner=env.step(action,player)
                next_avail=env.available_actions() if not done else []
                if done:
                    agent.update(state,action,reward,next_state,next_avail,done)
                    if winner==1: stats['wins']+=1
                    elif winner==-1: stats['losses']+=1
                    else: stats['draws']+=1
                else:
                    agent.update(state,action,0,next_state,next_avail,done)
                history.append((state,action))
                state=next_state
                player=-1
            else:
                avail=env.available_actions()
                if opponent=='random':
                    opp_action=random.choice(avail)
                elif opponent=='self':
                    opp_action=agent.choose_action(state,avail,explore=False)
                else:
                    raise ValueError('Unknown opponent')
                next_state,reward,done,winner=env.step(opp_action,player)
                if done and winner==-1 and history:
                    last_state,last_action=history[-1]
                    agent.update(last_state,last_action,-1,next_state,[],True)
                    stats['losses']+=1
                state=next_state
                player=1
        if ep%1000==0 and agent.epsilon>0.01:
            agent.epsilon*=0.9
        if ep%(episodes//10 or 1)==0:
            total=stats['wins']+stats['losses']+stats['draws']
            win_rate=stats['wins']/total if total else 0
            print(f"Episode {ep}/{episodes} - win_rate: {win_rate:.3f} (wins={stats['wins']}, draws={stats['draws']}, losses={stats['losses']})")
    print('Training complete.')
    return agent

# ---------------- Evaluation Function ----------------
def evaluate(agent:QLearningAgent,games:int=2000,opponent:str='random'):
    env=TicTacToe()
    stats={'wins':0,'losses':0,'draws':0}
    for _ in range(games):
        state=env.reset()
        done=False
        player=1
        while not done:
            if player==1:
                action=agent.choose_action(state,env.available_actions(),explore=False)
                state,reward,done,winner=env.step(action,player)
                if done:
                    if winner==1: stats['wins']+=1
                    elif winner==-1: stats['losses']+=1
                    else: stats['draws']+=1
                player=-1
            else:
                avail=env.available_actions()
                if opponent=='random':
                    opp_action=random.choice(avail)
                elif opponent=='self':
                    opp_action=agent.choose_action(state,avail,explore=False)
                else:
                    raise ValueError('Unknown opponent')
                state,reward,done,winner=env.step(opp_action,player)
                if done:
                    if winner==-1: stats['losses']+=1
                    elif winner==1: stats['wins']+=1
                    else: stats['draws']+=1
                player=1
    total=games
    win_rate=stats['wins']/total
    draw_rate=stats['draws']/total
    loss_rate=stats['losses']/total
    print(f"Evaluation against {opponent}: {games} games -> wins: {stats['wins']} ({win_rate:.3f}), draws: {stats['draws']} ({draw_rate:.3f}), losses: {stats['losses']} ({loss_rate:.3f})")
    return stats

# ---------------- Main ----------------
if __name__=='__main__':
    TRAIN=True
    EPISODES=20000
    SAVE_FILE='q_agent.pkl'

    agent=QLearningAgent(alpha=0.3,gamma=0.9,epsilon=0.2)

    if TRAIN:
        print('Starting training...')
        trained_agent=train(agent,episodes=EPISODES,opponent='random')
        trained_agent.save(SAVE_FILE)
        print(f'Saved trained Q-table to {SAVE_FILE}')
    else:
        print('Loading agent from file...')
        agent.load(SAVE_FILE)
        trained_agent=agent

    evaluate(trained_agent,games=2000,opponent='random')
    evaluate(trained_agent,games=2000,opponent='self')

    print('Done.')


Starting training...
Episode 2000/20000 - win_rate: 0.645 (wins=1290, draws=214, losses=496)
Episode 4000/20000 - win_rate: 0.684 (wins=2737, draws=417, losses=846)
Episode 6000/20000 - win_rate: 0.717 (wins=4300, draws=585, losses=1115)
Episode 8000/20000 - win_rate: 0.734 (wins=5874, draws=780, losses=1346)
Episode 10000/20000 - win_rate: 0.749 (wins=7490, draws=972, losses=1538)
Episode 12000/20000 - win_rate: 0.763 (wins=9151, draws=1151, losses=1698)
Episode 14000/20000 - win_rate: 0.773 (wins=10824, draws=1307, losses=1869)
Episode 16000/20000 - win_rate: 0.782 (wins=12519, draws=1472, losses=2009)
Episode 18000/20000 - win_rate: 0.792 (wins=14260, draws=1614, losses=2126)
Episode 20000/20000 - win_rate: 0.801 (wins=16025, draws=1764, losses=2211)
Training complete.
Saved trained Q-table to q_agent.pkl
Evaluation against random: 2000 games -> wins: 1748 (0.874), draws: 179 (0.089), losses: 73 (0.036)
Evaluation against self: 2000 games -> wins: 1742 (0.871), draws: 178 (0.089), l